# Aula 23: Protocolo de Árvore Geradora (STP / RSTP) — Tolerância a Falhas na Rede RF/Ethernet do Visão-AGV

## 1. Visão Geral da Aula

Nesta aula, simularemos o funcionamento do **Spanning Tree Protocol (STP)** e do **Rapid Spanning Tree Protocol (RSTP)** aplicados à infraestrutura de comunicação sem fio e cabeada do sistema **Visão-AGV**.

### Contexto no Projeto Visão-AGV
Para garantir que o robô não perca conexão com a central SCADA/MES durante falhas físicas em Access Points (APs) ou switches da fábrica, a topologia de rede conta com **links redundantes**.
No entanto, links redundantes em redes Ethernet criam **loops de Camada 2**, gerando tempestades de broadcast (*broadcast storms*) que paralisam a telemetria do AGV.

**Objetivos do STP/RSTP:**
1. Identificar e eleger a **Bridge Raiz (Root Bridge)** com base na menor prioridade/MAC.
2. Calcular o **Custo de Caminho até o Raiz (Root Path Cost)** para cada switch.
3. Definir papéis das portas: **Root Port (RP)**, **Designated Port (DP)** e **Blocking/Alternate Port (AP)**.
4. Bloquear links redundantes em operação normal e reativá-los automaticamente em caso de falha de hardware.


In [ ]:
# Estrutura de Simulação de Switch de Redes da Fábrica (Bridge STP)

class SwitchSTP:
    """
    Representa um switch ou access point na malha de comunicação do AGV.
    """
    def __init__(self, switch_id, prioridade, mac_address):
        self.switch_id = switch_id          # Identificador (ex: 'SW-CORE')
        self.prioridade = prioridade        # Prioridade STP (ex: 32768, menor vence)
        self.mac_address = mac_address      # Endereço MAC físico
        self.bridge_id = (prioridade, mac_address)
        
        # Estado STP
        self.root_bridge_id = self.bridge_id # Inicialmente assume que é o raiz
        self.custo_ate_raiz = 0
        self.portas = {}                    # {porta_id: {'vizinho': id, 'custo': int, 'estado': 'FORWARDING'/'BLOCKING'}}

    def adicionar_link(self, porta_id, vizinho_id, custo_link=4):
        """Conecta uma porta física a outro switch com determinado custo de link (ex: 1Gbps = custo 4)."""
        self.portas[porta_id] = {
            'vizinho': vizinho_id,
            'custo': custo_link,
            'papel': 'DESIGNATED',
            'estado': 'FORWARDING'
        }

    def __repr__(self):
        return f"<{self.switch_id} | BID: {self.bridge_id} | RaizConhecido: {self.root_bridge_id[1]}>"

print("Classe SwitchSTP pronta!")

## 2. Eleição da Bridge Raiz (Root Bridge) e Cálculo de Custo

Na inicialização da rede da fábrica, todos os switches trocam quadros BPDU (Bridge Protocol Data Unit) para determinar quem é o líder central da topologia.

In [ ]:
# Simulação do Algoritmo de Eleição do STP

switches = {
    'SW-CORE': SwitchSTP('SW-CORE', prioridade=4096, mac_address='00:11:22:33:44:00'),    # Central SCADA
    'AP-ZONA-A': SwitchSTP('AP-ZONA-A', prioridade=32768, mac_address='00:11:22:33:44:11'), # Setor Almacenagem
    'AP-ZONA-B': SwitchSTP('AP-ZONA-B', prioridade=32768, mac_address='00:11:22:33:44:22'), # Células Manufatura
    'SW-DOCAS': SwitchSTP('SW-DOCAS', prioridade=32768, mac_address='00:11:22:33:44:33')  # Zona de Carga
}

# Configuração dos Links Redundantes em Anel (Forma um Loop de Rede sem STP)
switches['SW-CORE'].adicionar_link('P1', 'AP-ZONA-A', custo_link=4)
switches['SW-CORE'].adicionar_link('P2', 'AP-ZONA-B', custo_link=4)

switches['AP-ZONA-A'].adicionar_link('P1', 'SW-CORE', custo_link=4)
switches['AP-ZONA-A'].adicionar_link('P2', 'SW-DOCAS', custo_link=4)

switches['AP-ZONA-B'].adicionar_link('P1', 'SW-CORE', custo_link=4)
switches['AP-ZONA-B'].adicionar_link('P2', 'SW-DOCAS', custo_link=4)

switches['SW-DOCAS'].adicionar_link('P1', 'AP-ZONA-A', custo_link=4)
switches['SW-DOCAS'].adicionar_link('P2', 'AP-ZONA-B', custo_link=4)

def eleger_root_bridge(rede):
    """Determina a Bridge Raiz da rede com base na menor (Prioridade, MAC)."""
    root_vencedor = min(rede.values(), key=lambda sw: sw.bridge_id)
    for sw in rede.values():
        sw.root_bridge_id = root_vencedor.bridge_id
    return root_vencedor

root = eleger_root_bridge(switches)
print(f"👑 ROOT BRIDGE ELEITA PARA O VISÃO-AGV: {root.switch_id} (BID: {root.bridge_id})")

## 3. Bloqueio de Portas e Eliminação de Loops (RSTP)

Identificamos o link com maior custo acumulado/ID para ser colocado em estado `BLOCKING` (Porta Alternativa), garantindo topologia em árvore sem loops de tempestade.

In [ ]:
def aplicar_estado_portas_stp(rede, root_id):
    """
    Simula o bloqueio lógico do link entre AP-ZONA-B e SW-DOCAS para quebrar o loop do anel.
    """
    print("=== ESTADO FINAL DAS PORTAS STP DA REDE DO AGV ===\n")
    
    # Bloqueio estratégico do link secundário AP-ZONA-B <-> SW-DOCAS
    rede['SW-DOCAS'].portas['P2']['papel'] = 'ALTERNATE (AP)'
    rede['SW-DOCAS'].portas['P2']['estado'] = 'BLOCKING'
    
    for sw_name, sw in rede.items():
        print(f"Switch/AP: {sw_name} {'[ROOT]' if sw.bridge_id == root_id else ''}")
        for porta, info in sw.portas.items():
            print(f"  ├─ Porta {porta} -> Conectado a {info['vizinho']:12s} | Papel: {info['papel']:15s} | Estado: {info['estado']}")
        print()

aplicar_estado_portas_stp(switches, root.bridge_id)

## 4. Análise de Convergência no Projeto Visão-AGV

1. **Segurança contra Tempestades de Broadcast:** Sem o STP/RSTP, quadros de telemetria UDP do AGV entrariam em *loop* infinito entre os Access Points, derrubando o servidor SCADA.
2. **Convergência Rápida (RSTP):** Enquanto o STP tradicional leva de 30 a 50 segundos para reagir a rompimentos de cabos, o **RSTP (IEEE 802.1w)** restabelece links alternativos em < 2 segundos, evitando paradas de emergência (*E-stop*) do AGV por perda de *heartbeat*.